# G12: quantized policy for GRPO rollouts (Kaggle)

For each bit-width: the quantized model draws completions at temperature 1; the bf16 model scores the same tokens.
Reported per format: rollout tokens/s, the share of tokens outside objective_v1's trust region [1-eps, 1+eps],
|log-ratio|, k3, mean KL(bf16 || quant) and top-1 agreement (same fields as G10's `rl-loop/mismatch_probe.py`).
Stage 1 is the quantization gap against llama.cpp's bf16; stage 2 (PyTorch) is the gap G10's trainer sees. `bf16` in `FORMATS`
adds a bf16 sampler row: its decode tok/s is the baseline of `tok_s_vs_bf16`, and in stage 2 its mismatch is the
llama.cpp-versus-PyTorch gap with no quantization in it. Compare tok/s only inside one session (the GPU varies between sessions).

**Smoke run first:** keep `N = 4`, `K = 2`, `MAX_NEW = 32` and `FORMATS = ["Q8_0"]`; check that a `tok/s` line and an
`outside` value print for it. Then set the real values. The first run builds llama-cpp-python with CUDA (~15-30 min).

Kaggle: Accelerator **GPU T4 x2**, Internet **on**. Use **Save Version, Save & Run All (Commit)**. No token is needed
(both models are not gated).

In [ ]:
# --- settings -----------------------------------------------------------------------------------------------
FORMATS   = ["Q8_0"]           # <-- samplers to test: bf16, Q8_0, Q6_K, Q5_K_M, Q4_K_M, Q3_K_M. bf16 as a sampler is the decode-speed
                               #     baseline (tok/s) and, in stage 2, the llama.cpp-versus-PyTorch gap; always scored against the bf16 reference
RUN_BY    = ""                 # <-- your GitHub handle, recorded with the results
PROMPTS   = "first"            # "first" = the first N of the 100 fixed prompts (the first 48 are 40 English + 8 Indic, no code);
                               # "balanced" = N taken round-robin over english / indic / code / tool_call / reasoning (10 each at N = 50)
N         = 4                  # prompts (of the 100 fixed prompts)           real run: 48
K         = 2                  # completions per prompt                       real run: 8
MAX_NEW   = 32                 # new tokens per completion                    real run: 256
EPS       = 0.2                # objective_v1's trust region (grpo.TR_EPSILON)
SEED      = 1234
N_CTX     = 2048
BASE      = "bf16"
MODEL, REVISION = "sarvamai/sarvam-1", "e9607337286ddf496d4a2562b194e489dcf3feea"
# or Granite: MODEL, REVISION = "ibm-granite/granite-3.0-3b-a800m-base", "142d2c8e3f7fb8844011cfa5565f0e520f0824b9"
# second models Team 10 suggested (pin a revision before the real run): "Qwen/Qwen2.5-Coder-1.5B", "HuggingFaceTB/SmolLM2-1.7B"
TORCH_REFERENCE = True         # stage 2: score the same tokens with the HF model in PyTorch (Team 10's trainer setup)
TORCH_DTYPE = "bfloat16"       # Team 10's trainer: bf16 weights and autocast, fp32 logits. T4 has no native bf16: if stage 2 is slow, "float32"
TORCH = "torch==2.11.0"        # CUDA 12.6 build, from the PyTorch index below
TORCH_INDEX = "https://download.pytorch.org/whl/cu126"
LLAMA_TAG = "b11669"
LLAMA_CPP_PYTHON = "0.3.36"
DRIVE_DIR = "/content/drive/MyDrive/era-v5-g12/rollout_mismatch"
BF16_FALLBACK_GPU_LAYERS = 12  # reference GPU layers, only if bf16 does not fit next to the sampler on the GPUs
LIMIT = None                   # unused here; kept because the shared setup cells record it
NOTEBOOK_VERSION = "1edb192ad957"
assert FORMATS and all(f in {"bf16", "Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q3_K_M"} for f in FORMATS), FORMATS
assert PROMPTS in ("first", "balanced"), PROMPTS

In [ ]:
import datetime, glob, json, os, platform, shutil, subprocess, sys, time
LOCAL_TEST = bool(os.environ.get("EVAL_MINI_LOCAL_TEST"))
IN_KAGGLE = os.path.isdir("/kaggle/working")
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if LOCAL_TEST:
    LOCAL_OUT, WORK = os.environ.get("EVAL_MINI_OUT", "/tmp/eval_mini_local"), os.path.expanduser("~/.cache/era-v5-quant")
elif IN_KAGGLE:
    LOCAL_OUT, WORK = "/kaggle/working/eval_mini_out", "/kaggle/tmp/eval_mini_work"   # output vs big files
else:
    LOCAL_OUT, WORK = "/content/eval_mini_out", "/content/work"
OUT = LOCAL_OUT
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = DRIVE_DIR
    except Exception as e:  # e.g. not available from an editor-attached runtime
        print(f"Drive not mounted ({e!r}); output goes to {LOCAL_OUT}. DOWNLOAD THE ZIP AT THE END.")
for d in (OUT, f"{OUT}/logs", WORK):
    os.makedirs(d, exist_ok=True)
CODE = f"{WORK}/eval_mini_code"; os.makedirs(CODE, exist_ok=True)

def sh(cmd, log=None):
    """Run a shell command; stream its output (and append it to a log); stop the notebook if it fails."""
    print("$", cmd, flush=True)
    full = f"set -o pipefail; ( {cmd} ) 2>&1" + (f" | tee -a {log}" if log else "")
    rc = subprocess.run(full, shell=True, executable="/bin/bash").returncode
    if rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}" + (f"; see {log}" if log else ""))

PLATFORM = "local-test" if LOCAL_TEST else "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "other"
ENV = {"started": datetime.datetime.now().isoformat(timespec="seconds"), "platform": PLATFORM,
       "output_dir": OUT, "run_by": RUN_BY, "notebook_version": NOTEBOOK_VERSION,
       "settings": {k: globals()[k] for k in ("LIMIT", "FORMATS", "BASE", "MODEL", "REVISION",
       "LLAMA_TAG", "LLAMA_CPP_PYTHON", "BF16_FALLBACK_GPU_LAYERS")}, "python": platform.python_version(), "formats": {}}
def save_env():
    json.dump(ENV, open(f"{OUT}/environment.json", "w"), indent=1)
if shutil.which("nvidia-smi"):
    ENV["gpu"] = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
                                shell=True, capture_output=True, text=True).stdout.strip()
    sh("nvidia-smi")
sh(f"df -h {WORK} | tail -1")
if not LOCAL_TEST:   # Kaggle needs Internet switched on in the notebook settings
    sh("curl -fsS -o /dev/null https://huggingface.co || (echo 'NO INTERNET: turn it on in the notebook settings' && false)")
save_env()
print("platform:", PLATFORM, "| output:", OUT, "| model files:", WORK)

In [ ]:
# the code and the 100 fixed prompts, embedded by build_notebook.py
SOURCES = {
"quant_rollout_mismatch.py": "#!/usr/bin/env python3\n\"\"\"Quantized policy for GRPO rollouts: speed and mismatch per bit-width against bf16 (G12 design task 3, with G10).\n\n    python quant_rollout_mismatch.py --ref model-bf16.gguf --quant Q8_0=model-Q8_0.gguf --quant Q6_K=model-Q6_K.gguf \\\\\n        --prompts ../prompts/fixed_100.jsonl --out rollout_mismatch.json [--n 48] [--k 8] [--max-new 256] [--eps 0.2]\n\nThe question G10's trainer asks of a sampler (rl-loop/mismatch_probe.py): objective_v1 keeps a token only when\nratio = trainer / sampler lies in [1 - eps, 1 + eps]. With a quantized sampler the ratio differs from 1 by the\nquantization error. Here the sampler is a quantized GGUF and the \"trainer\" is the bf16 GGUF of the same checkpoint:\n\n  1. the quantized model draws k completions per prompt at temperature 1 (top-k, top-p and min-p off, so the sampled\n     distribution is the model's own softmax); the decode speed is timed (rollout tokens/s);\n  2. both models score the same token ids, teacher-forced, with their full logits;\n  3. per sampled token: log_ratio = log p_bf16 - log p_quant; per position: KL(bf16 || quant) over the whole\n     vocabulary and whether the two models' top-1 tokens agree.\n\nReported per format (same numbers as mismatch_probe.py, so the two reports can be read side by side): share of tokens\noutside [1 - eps, 1 + eps] (below, above), mean and quantiles of |log_ratio|, k3 = mean(r - log r - 1), plus mean KL,\ntop-1 agreement, and the tokens/s of the rollout. Aggregate numbers only: no prompt or completion text is written.\n\nNot the same as G10's number: bf16 here is llama.cpp's bf16, not the torch trainer's fp32-master / bf16 autocast, so the\nengine-to-engine gap of G10's probe is not in these numbers, only the quantization gap (with llama.cpp's own bf16\narithmetic as the floor). `--quant bf16=<same file>` is a determinism check, not a noise floor: the teacher-forced\nlogits of one file are bit-identical on every run, so it must report exactly 0 (any other value means a bug).\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse, json, math, sys, time\nfrom pathlib import Path\n\nimport numpy as np\n\nP_BUCKETS = ((0.0, 0.1), (0.1, 0.5), (0.5, 0.9), (0.9, 1.0001))   # by the sampler's probability, as mismatch_probe.py\nPOS_BUCKETS = ((0, 64), (64, 256), (256, 10**9))\nEPS = 0.2                                                           # grpo.TR_EPSILON; override with --eps\n\n\ndef log_softmax(x: np.ndarray) -> np.ndarray:\n    x = x.astype(np.float64)\n    m = x.max(axis=-1, keepdims=True)\n    return x - m - np.log(np.exp(x - m).sum(axis=-1, keepdims=True))\n\n\ndef position_stats(ref_logits: np.ndarray, q_logits: np.ndarray, tokens: np.ndarray, chunk: int = 64) -> dict:\n    \"\"\"ref_logits, q_logits: [T, V], row t predicts tokens[t]. Returns arrays of length T (float64 inside, in chunks).\"\"\"\n    lp_ref, lp_q, kl, agree, top_ref, top_q = [], [], [], [], [], []\n    for i in range(0, len(tokens), chunk):\n        a, b = log_softmax(ref_logits[i:i + chunk]), log_softmax(q_logits[i:i + chunk])\n        t = tokens[i:i + chunk]\n        rows = np.arange(len(t))\n        lp_ref.append(a[rows, t]); lp_q.append(b[rows, t])\n        kl.append((np.exp(a) * (a - b)).sum(axis=-1))\n        top_ref.append(a.argmax(-1)); top_q.append(b.argmax(-1))\n        agree.append(top_ref[-1] == top_q[-1])\n    return {\"lp_ref\": np.concatenate(lp_ref), \"lp_q\": np.concatenate(lp_q),\n            \"kl\": np.concatenate(kl), \"agree\": np.concatenate(agree),\n            \"top1_ref\": np.concatenate(top_ref), \"top1_q\": np.concatenate(top_q)}\n\n\ndef quantiles(a: np.ndarray) -> list[float]:\n    return [float(f\"{x:.4g}\") for x in np.quantile(a, [0.5, 0.9, 0.99, 0.999])]\n\n\ndef ratio_stats(log_ratio: np.ndarray, eps: float) -> dict:\n    \"\"\"Same fields as mismatch_probe.ratio_stats; log_ratio = log(trainer) - log(sampler) over real tokens.\"\"\"\n    n = log_ratio.size\n    if n == 0:\n        return {\"tokens\": 0}\n    r, a = np.exp(log_ratio), np.abs(log_ratio)\n    below, above = r < 1 - eps, r > 1 + eps\n    return {\"tokens\": int(n), \"outside\": float((below | above).mean()), \"below\": float(below.mean()),\n            \"above\": float(above.mean()), \"mean_abs_logratio\": float(a.mean()),\n            \"abs_logratio_p50_p90_p99_p999\": quantiles(a), \"max_abs_logratio\": float(a.max()),\n            \"k3\": float((r - log_ratio - 1).mean())}\n\n\ndef cluster_bootstrap(per_pos: dict, clusters: np.ndarray, eps: float = EPS, n_boot: int = 2000, seed: int = 0) -> dict:\n    \"\"\"95% percentile intervals of the share outside [1 - eps, 1 + eps], mean |log-ratio|, k3, top-1 agreement (and KL when\n    present), resampling whole clusters (prompts) with replacement: the 8 rollouts of a prompt are not independent.\"\"\"\n    lr = per_pos[\"lp_ref\"] - per_pos[\"lp_q\"]\n    r = np.exp(lr)\n    cols = {\"outside\": ((r < 1 - eps) | (r > 1 + eps)).astype(float), \"mean_abs_logratio\": np.abs(lr),\n            \"k3\": r - lr - 1, \"top1_agreement\": per_pos[\"agree\"].astype(float)}\n    if \"kl\" in per_pos:\n        cols[\"mean_kl_ref_to_quant\"] = per_pos[\"kl\"].astype(float)\n    ids, inv = np.unique(clusters, return_inverse=True)\n    n = np.bincount(inv, minlength=len(ids)).astype(float)\n    sums = {k: np.bincount(inv, weights=v, minlength=len(ids)) for k, v in cols.items()}\n    pick = np.random.default_rng(seed).integers(0, len(ids), (n_boot, len(ids)))\n    tot = n[pick].sum(1)\n    out = {}\n    for k, s in sums.items():\n        est = s[pick].sum(1) / tot\n        lo, hi = np.percentile(est, [2.5, 97.5])\n        out[k] = [float(f\"{lo:.4g}\"), float(f\"{hi:.4g}\")]\n    out[\"clusters\"] = int(len(ids))\n    return out\n\n\ndef aggregate(per_pos: dict, positions: np.ndarray, eps: float = EPS, clusters: np.ndarray | None = None) -> dict:\n    \"\"\"per_pos: concatenated position_stats arrays over all rollouts; positions: index of each token in its completion.\"\"\"\n    log_ratio = per_pos[\"lp_ref\"] - per_pos[\"lp_q\"]\n    out = ratio_stats(log_ratio, eps)\n    if not out[\"tokens\"]:\n        return out\n    out[\"top1_agreement\"] = float(per_pos[\"agree\"].mean())\n    if \"kl\" in per_pos:\n        out[\"mean_kl_ref_to_quant\"] = float(per_pos[\"kl\"].mean())\n    p = np.exp(per_pos[\"lp_q\"])\n    out[\"by_sampler_prob\"] = {f\"{lo}-{min(hi, 1.0)}\": ratio_stats(log_ratio[(p >= lo) & (p < hi)], eps)\n                              for lo, hi in P_BUCKETS if ((p >= lo) & (p < hi)).any()}\n    out[\"by_position\"] = {f\"{lo}-{hi if hi < 10**9 else 'end'}\": ratio_stats(log_ratio[(positions >= lo) & (positions < hi)], eps)\n                          for lo, hi in POS_BUCKETS if ((positions >= lo) & (positions < hi)).any()}\n    if clusters is not None:\n        out[\"ci95_by_prompt\"] = cluster_bootstrap(per_pos, clusters, eps)\n    return out\n\n\ndef load_prompt_rows(path: Path) -> list[tuple[str, str]]:\n    \"\"\"(text, category) per prompt: jsonl with a \"prompt\", \"statement\" or \"question\" field (and an optional \"category\"),\n    or one prompt per line (category \"\").\"\"\"\n    out = []\n    for line in path.read_text(encoding=\"utf-8\").splitlines():\n        line = line.strip()\n        if not line:\n            continue\n        try:\n            row = json.loads(line)\n            out.append((next(row[k] for k in (\"prompt\", \"statement\", \"question\") if k in row), str(row.get(\"category\", \"\"))))\n        except (json.JSONDecodeError, StopIteration, TypeError, AttributeError):\n            out.append((line, \"\"))\n    return out\n\n\ndef select_prompts(rows: list[tuple[str, str]], n: int, balanced: bool = False) -> list[tuple[str, str]]:\n    \"\"\"The first n prompts, or with `balanced` n prompts taken round-robin across the categories (in file order within a\n    category), so a file that lists its categories one after another (fixed_100.jsonl: 40 English first) is not read as one.\"\"\"\n    if not balanced:\n        return rows[:n]\n    by_cat: dict[str, list] = {}\n    for r in rows:\n        by_cat.setdefault(r[1], []).append(r)\n    out, i = [], 0\n    while len(out) < n and any(i < len(v) for v in by_cat.values()):\n        out += [v[i] for v in by_cat.values() if i < len(v)][:n - len(out)]\n        i += 1\n    return out\n\n\ndef load_prompts(path: Path, n: int, balanced: bool = False) -> list[str]:\n    return [text for text, _ in select_prompts(load_prompt_rows(path), n, balanced)]\n\n\nclass LlamaEngine:\n    \"\"\"One GGUF through llama-cpp-python (the build eval_wide uses): sampling with timing, and teacher-forced logits.\"\"\"\n\n    def __init__(self, path: str, n_ctx: int = 2048, n_gpu_layers: int = -1, seed: int = 1234):\n        from llama_cpp import Llama\n        self.llm = Llama(model_path=path, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, logits_all=True, seed=seed, verbose=False)\n        self.n_ctx, self.eos = n_ctx, self.llm.token_eos()\n\n    def tokenize(self, text: str) -> list[int]:\n        return self.llm.tokenize(text.encode(\"utf-8\"), add_bos=True, special=True)\n\n    def sample(self, prompt_ids: list[int], max_new: int) -> tuple[list[int], float]:\n        \"\"\"Temperature 1 from the model's own softmax. Returns (new token ids, seconds spent decoding).\"\"\"\n        self.llm.reset()\n        out, t0 = [], time.perf_counter()\n        for tok in self.llm.generate(prompt_ids, temp=1.0, top_k=0, top_p=1.0, min_p=0.0, repeat_penalty=1.0, reset=True):\n            if tok == self.eos or len(out) >= max_new or len(prompt_ids) + len(out) >= self.n_ctx - 1:\n                break\n            out.append(tok)\n        return out, time.perf_counter() - t0\n\n    def logits(self, ids: list[int]) -> np.ndarray:\n        \"\"\"[len(ids), V] float32; row j predicts ids[j + 1].\"\"\"\n        self.llm.reset()\n        self.llm.eval(ids)\n        return np.array(self.llm.scores[:len(ids)], dtype=np.float32)\n\n\ndef run_format(name: str, sampler, ref, prompts: list[list[int]], k: int, max_new: int, eps: float, dump_path=None,\n               categories: list[str] | None = None) -> dict:\n    parts, positions, clusters, new_tokens, seconds, empty = [], [], [], 0, 0.0, 0\n    dump = open(dump_path, \"w\") if dump_path else None\n    for pi, p in enumerate(prompts):\n        for _ in range(k):\n            comp, secs = sampler.sample(p, max_new)\n            new_tokens, seconds = new_tokens + len(comp), seconds + secs\n            if not comp:\n                empty += 1\n                continue\n            ids = p + comp\n            rows = slice(len(p) - 1, len(ids) - 1)                       # rows that predict the completion tokens\n            ps = position_stats(ref.logits(ids)[rows], sampler.logits(ids)[rows], np.array(comp))\n            parts.append(ps)\n            positions.append(np.arange(len(comp)))\n            clusters.append(np.full(len(comp), pi))\n            if dump:   # token ids and the quantized model's numbers, for a second reference (score_with_torch.py)\n                dump.write(json.dumps({\"format\": name, \"prompt_idx\": pi, \"prompt_len\": len(p), \"category\": (categories[pi] if categories else None), \"ids\": ids,\n                                       \"lp_q\": [round(float(x), 6) for x in ps[\"lp_q\"]],\n                                       \"top1_q\": [int(x) for x in ps[\"top1_q\"]]}) + \"\\n\")\n    if dump:\n        dump.close()\n    keys = (\"lp_ref\", \"lp_q\", \"kl\", \"agree\", \"top1_ref\", \"top1_q\")\n    per_pos = {k_: np.concatenate([x[k_] for x in parts]) for k_ in keys} if parts else None\n    res = {\"format\": name, \"rollouts\": len(parts), \"empty_rollouts\": empty, \"new_tokens\": new_tokens,\n           \"decode_seconds\": round(seconds, 2), \"rollout_tokens_per_s\": round(new_tokens / seconds, 2) if seconds else None}\n    if per_pos:\n        res.update(aggregate(per_pos, np.concatenate(positions), eps, np.concatenate(clusters)))\n    return res\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--ref\", required=True, help=\"bf16 GGUF (the reference, scores every sampled token)\")\n    ap.add_argument(\"--quant\", action=\"append\", required=True, metavar=\"NAME=PATH\", help=\"a sampler GGUF; repeat per bit-width\")\n    ap.add_argument(\"--prompts\", required=True, type=Path)\n    ap.add_argument(\"--out\", required=True, type=Path)\n    ap.add_argument(\"--n\", type=int, default=48, help=\"prompts\")\n    ap.add_argument(\"--balanced\", action=\"store_true\", help=\"take the prompts round-robin across the file's categories, not the first n\")\n    ap.add_argument(\"--k\", type=int, default=8, help=\"completions per prompt\")\n    ap.add_argument(\"--max-new\", type=int, default=256)\n    ap.add_argument(\"--n-ctx\", type=int, default=2048)\n    ap.add_argument(\"--n-gpu-layers\", type=int, default=-1)\n    ap.add_argument(\"--ref-n-gpu-layers\", type=int, default=None, help=\"for the bf16 reference only (default: --n-gpu-layers)\")\n    ap.add_argument(\"--eps\", type=float, default=EPS)\n    ap.add_argument(\"--seed\", type=int, default=1234)\n    ap.add_argument(\"--dump-dir\", type=Path, default=None,\n                    help=\"write <format>_rollouts.jsonl (token ids, the sampler's log-probabilities and top-1) for score_with_torch.py\")\n    a = ap.parse_args()\n    formats = dict(q.split(\"=\", 1) for q in a.quant)\n    ref = LlamaEngine(a.ref, a.n_ctx, a.n_gpu_layers if a.ref_n_gpu_layers is None else a.ref_n_gpu_layers, a.seed)\n    chosen = select_prompts(load_prompt_rows(a.prompts), a.n, a.balanced)\n    tokenized = [(ref.tokenize(text), cat) for text, cat in chosen]\n    tokenized = [(p, cat) for p, cat in tokenized if len(p) < a.n_ctx - a.max_new - 1]\n    prompts = [p for p, _ in tokenized]\n    categories: dict[str, int] = {}\n    for _, cat in tokenized:\n        categories[cat or \"-\"] = categories.get(cat or \"-\", 0) + 1\n    results = []\n    for i, (name, path) in enumerate(formats.items()):\n        sampler = LlamaEngine(path, a.n_ctx, a.n_gpu_layers, a.seed + 1 + i)\n        t0 = time.time()\n        if a.dump_dir:\n            a.dump_dir.mkdir(parents=True, exist_ok=True)\n        results.append(run_format(name, sampler, ref, prompts, a.k, a.max_new, a.eps,\n                                  a.dump_dir / f\"{name}_rollouts.jsonl\" if a.dump_dir else None, [c for _, c in tokenized]))\n        print(f\"{name}: {results[-1].get('rollout_tokens_per_s')} tok/s, outside [{1 - a.eps:.2f}, {1 + a.eps:.2f}] \"\n              f\"{results[-1].get('outside')}, top-1 {results[-1].get('top1_agreement')}, {time.time() - t0:.0f}s\", flush=True)\n        del sampler\n    a.out.write_text(json.dumps({\"reference\": Path(a.ref).name, \"prompts\": len(prompts), \"prompt_selection\": \"balanced\" if a.balanced else \"first\",\n                                 \"prompt_categories\": categories, \"k\": a.k, \"max_new\": a.max_new,\n                                 \"eps\": a.eps, \"temperature\": 1.0, \"seed\": a.seed, \"formats\": results}, indent=1))\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
"score_with_torch.py": "#!/usr/bin/env python3\n\"\"\"Second reference for the quantized-rollout mismatch: score the same sampled tokens with PyTorch (Team 10's setup).\n\n    python score_with_torch.py --dump out/dump/Q8_0_rollouts.jsonl [--dump ...] --model sarvamai/sarvam-1 \\\\\n        --revision <commit> --out torch_reference.json [--dtype bfloat16] [--device cuda] [--eps 0.2]\n\nStage 1 (`quant_rollout_mismatch.py --dump-dir`, llama.cpp) samples with a quantized GGUF and writes, per rollout, the token\nids, the quantized model's log-probability of every sampled token and its top-1 token per position. This stage loads the\nHugging Face model, runs the same ids through it, and reports what Team 10's trainer would see: log_ratio =\nlog p_torch - log p_sampler for every sampled token, the share outside [1 - eps, 1 + eps], |log-ratio| quantiles, k3 (the\nfields of rl-loop/mismatch_probe.py), top-1 agreement, with 95% intervals that resample prompts (the rollouts of a prompt\nare not independent).\n\nTeam 10's trainer (rl-loop/policy.py) keeps frozen weights in bf16, runs bf16 autocast, takes the logits in fp32 and\napplies log_softmax: that is `--dtype bfloat16` here. `--dtype float32` runs the whole model in fp32 (as mismatch_probe.py\n--hf-dtype float32). On a T4 (no native bf16) check `torch_bf16_supported` in the output and prefer float32 if bf16 is slow.\n\nMean KL(torch || quant) is not reported: it needs the quantized model's full distribution, which the dump does not keep\n(the llama.cpp reference mode of quant_rollout_mismatch.py reports it). The dump holds token ids of model-generated\ncompletions: keep it out of the repository; only the aggregate output of this script is committed.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse, json, sys, time\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nimport quant_rollout_mismatch as qrm  # noqa: E402\n\n\ndef load_dump(path: Path) -> list[dict]:\n    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]\n\n\ndef categories_from_prompt_file(path: Path, n: int, balanced: bool) -> list[str]:\n    \"\"\"Category of each prompt index, as stage 1 selected them (for dumps made before stage 1 recorded it).\"\"\"\n    rows = qrm.select_prompts(qrm.load_prompt_rows(path), n, balanced)\n    return [cat for _, cat in rows]\n\n\ndef score_records(model, records: list[dict], device, dtype_name: str) -> tuple[dict, np.ndarray, np.ndarray]:\n    \"\"\"Run every rollout's ids through `model`; returns per-token arrays (lp_ref, lp_q, agree, category), positions, clusters.\"\"\"\n    import torch\n    lp_ref, lp_q, agree, positions, clusters, cats = [], [], [], [], [], []\n    vocab = None\n    model.eval()\n    with torch.no_grad():\n        for rec in records:\n            if max(rec[\"ids\"]) >= model.config.vocab_size:\n                raise ValueError(f\"id {max(rec['ids'])} is outside the model's vocabulary ({model.config.vocab_size}): \"\n                                 \"wrong model or revision for this dump?\")\n            ids = torch.tensor([rec[\"ids\"]], device=device)\n            plen = rec[\"prompt_len\"]\n            logits = model(input_ids=ids).logits[0, plen - 1:len(rec[\"ids\"]) - 1].float()\n            vocab = logits.shape[-1]\n            comp = ids[0, plen:]\n            if int(comp.max()) >= vocab:\n                raise ValueError(f\"a sampled id is outside the output width ({vocab}): wrong model for this dump?\")\n            logp = torch.log_softmax(logits, -1)\n            lp_ref.append(logp.gather(-1, comp[:, None]).squeeze(-1).cpu().numpy().astype(np.float64))\n            agree.append((logp.argmax(-1).cpu().numpy() == np.asarray(rec[\"top1_q\"])))\n            lp_q.append(np.asarray(rec[\"lp_q\"], dtype=np.float64))\n            positions.append(np.arange(len(comp)))\n            clusters.append(np.full(len(comp), rec[\"prompt_idx\"]))\n            cats.append(np.full(len(comp), rec.get(\"category\") or \"-\", dtype=object))\n    per_pos = {\"lp_ref\": np.concatenate(lp_ref), \"lp_q\": np.concatenate(lp_q), \"agree\": np.concatenate(agree),\n               \"category\": np.concatenate(cats)}\n    return per_pos, np.concatenate(positions), np.concatenate(clusters)\n\n\ndef main() -> int:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--dump\", action=\"append\", required=True, type=Path, help=\"<format>_rollouts.jsonl; repeat per format\")\n    ap.add_argument(\"--model\", required=True); ap.add_argument(\"--revision\", default=None)\n    ap.add_argument(\"--out\", required=True, type=Path)\n    ap.add_argument(\"--dtype\", default=\"bfloat16\", choices=[\"bfloat16\", \"float32\", \"float16\"])\n    ap.add_argument(\"--device\", default=None)\n    ap.add_argument(\"--eps\", type=float, default=qrm.EPS)\n    ap.add_argument(\"--prompts-file\", type=Path, default=None, help=\"for dumps without a category per rollout: the prompt file stage 1 used\")\n    ap.add_argument(\"--n\", type=int, default=50, help=\"with --prompts-file: the --n of stage 1\")\n    ap.add_argument(\"--balanced\", action=\"store_true\", help=\"with --prompts-file: stage 1 used --balanced\")\n    a = ap.parse_args()\n    import torch\n    from transformers import AutoModelForCausalLM\n    device = torch.device(a.device or (\"cuda\" if torch.cuda.is_available() else \"cpu\"))\n    t0 = time.time()\n    model = AutoModelForCausalLM.from_pretrained(a.model, revision=a.revision, dtype=getattr(torch, a.dtype)).to(device)\n    out = {\"reference\": f\"{a.model}@{a.revision}\", \"dtype\": a.dtype, \"device\": str(device), \"torch\": torch.__version__,\n           \"torch_bf16_supported\": bool(device.type == \"cuda\" and torch.cuda.is_bf16_supported()), \"eps\": a.eps, \"formats\": []}\n    for path in a.dump:\n        records = load_dump(path)\n        if not records:\n            continue\n        name = records[0][\"format\"]\n        if a.prompts_file and not any(r.get(\"category\") for r in records):      # an older dump: derive the category from the prompt file\n            cat_of = categories_from_prompt_file(a.prompts_file, a.n, a.balanced)\n            for r in records:\n                r[\"category\"] = cat_of[r[\"prompt_idx\"]]\n        per_pos, positions, clusters = score_records(model, records, device, a.dtype)\n        res = {\"format\": name, \"rollouts\": len(records), \"new_tokens\": int(len(per_pos[\"lp_ref\"]))}\n        cat = per_pos.pop(\"category\")\n        res.update(qrm.aggregate(per_pos, positions, a.eps, clusters))\n        if set(cat) != {\"-\"}:\n            res[\"by_category\"] = {c: {k: v for k, v in qrm.aggregate({kk: vv[cat == c] for kk, vv in per_pos.items()}, positions[cat == c], a.eps,\n                                                                     clusters[cat == c]).items() if k not in (\"by_sampler_prob\", \"by_position\")}\n                                  for c in sorted(set(cat))}\n        out[\"formats\"].append(res)\n        ci = res[\"ci95_by_prompt\"]\n        print(f\"{name}: outside {res['outside']:.4f} {ci['outside']}, top-1 {res['top1_agreement']:.4f} {ci['top1_agreement']}, \"\n              f\"k3 {res['k3']:.5f}, {time.time() - t0:.0f}s\", flush=True)\n    a.out.write_text(json.dumps(out, indent=1))\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n",
"fixed_100.jsonl": "{\"id\": \"p001\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Explain in two sentences why the sky looks blue during the day.\"}\n{\"id\": \"p002\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Write a short, friendly email asking a colleague to move a meeting from Tuesday to Thursday.\"}\n{\"id\": \"p003\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"List three differences between a list and a tuple in Python.\"}\n{\"id\": \"p004\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Give me a packing checklist for a three-day trip to the mountains.\"}\n{\"id\": \"p005\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"What is the difference between weather and climate?\"}\n{\"id\": \"p006\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Summarize the plot of a story about a lighthouse keeper who finds a message in a bottle.\"}\n{\"id\": \"p007\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Suggest five names for a small neighbourhood bakery.\"}\n{\"id\": \"p008\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Explain what an API is to someone who has never programmed.\"}\n{\"id\": \"p009\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Write a haiku about monsoon rain.\"}\n{\"id\": \"p010\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"How do I politely decline a wedding invitation?\"}\n{\"id\": \"p011\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Describe how a bicycle stays upright while moving.\"}\n{\"id\": \"p012\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"What are good habits for sleeping better?\"}\n{\"id\": \"p013\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Explain compound interest with a simple example using round numbers.\"}\n{\"id\": \"p014\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Write two sentences that begin with the word 'Although'.\"}\n{\"id\": \"p015\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Give three tips for learning a new language as an adult.\"}\n{\"id\": \"p016\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"What does it mean when a function is called recursively?\"}\n{\"id\": \"p017\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Rewrite this sentence more formally: 'Hey, can you send me that file soon?'\"}\n{\"id\": \"p018\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Explain the difference between a virus and a bacterium.\"}\n{\"id\": \"p019\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Plan a simple vegetarian dinner for four people.\"}\n{\"id\": \"p020\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Why do we have leap years?\"}\n{\"id\": \"p021\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Write a limerick about a cat that learned to swim.\"}\n{\"id\": \"p022\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"What is photosynthesis? Answer in three short sentences.\"}\n{\"id\": \"p023\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Give two arguments for and two against working from home.\"}\n{\"id\": \"p024\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"How does a refrigerator keep food cold?\"}\n{\"id\": \"p025\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Write a polite reminder message for an unpaid invoice.\"}\n{\"id\": \"p026\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Explain what a hash table is and why lookups are fast.\"}\n{\"id\": \"p027\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"What should I check before buying a second-hand laptop?\"}\n{\"id\": \"p028\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Tell a two-sentence bedtime story about a sleepy elephant.\"}\n{\"id\": \"p029\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Explain the idea of supply and demand using the price of mangoes.\"}\n{\"id\": \"p030\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"If a train leaves at 9:15 and the trip takes 2 hours 50 minutes, when does it arrive?\"}\n{\"id\": \"p031\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Describe your ideal Sunday morning in four sentences.\"}\n{\"id\": \"p032\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"What are the main causes of traffic jams in large cities?\"}\n{\"id\": \"p033\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Explain what a neural network is in plain words, in under 80 words.\"}\n{\"id\": \"p034\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Write a thank-you note to a teacher.\"}\n{\"id\": \"p035\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Give me three ways to reduce plastic use at home.\"}\n{\"id\": \"p036\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"What is the difference between 'affect' and 'effect'?\"}\n{\"id\": \"p037\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Write a short product description for a steel water bottle.\"}\n{\"id\": \"p038\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Why is the Moon visible in the daytime sometimes?\"}\n{\"id\": \"p039\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Suggest a one-week beginner running plan.\"}\n{\"id\": \"p040\", \"category\": \"english\", \"lang\": \"en\", \"prompt\": \"Explain what inflation is and who it hurts most.\"}\n{\"id\": \"p041\", \"category\": \"indic\", \"lang\": \"hi\", \"prompt\": \"\u0905\u092a\u0928\u0947 \u0926\u094b\u0938\u094d\u0924 \u0915\u0947 \u0932\u093f\u090f \u091c\u0928\u094d\u092e\u0926\u093f\u0928 \u0915\u0940 \u090f\u0915 \u091b\u094b\u091f\u0940 \u0936\u0941\u092d\u0915\u093e\u092e\u0928\u093e \u0932\u093f\u0916\u093f\u090f\u0964\"}\n{\"id\": \"p042\", \"category\": \"indic\", \"lang\": \"hi\", \"prompt\": \"\u092c\u093e\u0930\u093f\u0936 \u0915\u093e \u092e\u094c\u0938\u092e \u0939\u092e\u093e\u0930\u0947 \u0932\u093f\u090f \u0915\u094d\u092f\u094b\u0902 \u091c\u093c\u0930\u0942\u0930\u0940 \u0939\u0948? \u0926\u094b \u0935\u093e\u0915\u094d\u092f\u094b\u0902 \u092e\u0947\u0902 \u092c\u0924\u093e\u0907\u090f\u0964\"}\n{\"id\": \"p043\", \"category\": \"indic\", \"lang\": \"hi\", \"prompt\": \"\u092d\u093e\u0930\u0924 \u0915\u0940 \u0930\u093e\u091c\u0927\u093e\u0928\u0940 \u0915\u094d\u092f\u093e \u0939\u0948, \u0914\u0930 \u0935\u0939 \u0915\u093f\u0938 \u0928\u0926\u0940 \u0915\u0947 \u092a\u093e\u0938 \u092c\u0938\u0940 \u0939\u0948?\"}\n{\"id\": \"p044\", \"category\": \"indic\", \"lang\": \"bn\", \"prompt\": \"\u09ac\u09a8\u09cd\u09a7\u09c1\u09b0 \u099c\u09a8\u09cd\u09ae\u09a6\u09bf\u09a8\u09c7\u09b0 \u099c\u09a8\u09cd\u09af \u098f\u0995\u099f\u09bf \u099b\u09cb\u099f \u09b6\u09c1\u09ad\u09c7\u099a\u09cd\u099b\u09be \u09ac\u09be\u09b0\u09cd\u09a4\u09be \u09b2\u09c7\u0996\u09cb\u0964\"}\n{\"id\": \"p045\", \"category\": \"indic\", \"lang\": \"bn\", \"prompt\": \"\u09ac\u09b0\u09cd\u09b7\u09be\u0995\u09be\u09b2 \u0986\u09ae\u09be\u09a6\u09c7\u09b0 \u099c\u09a8\u09cd\u09af \u0995\u09c7\u09a8 \u0997\u09c1\u09b0\u09c1\u09a4\u09cd\u09ac\u09aa\u09c2\u09b0\u09cd\u09a3? \u09a6\u09c1\u099f\u09bf \u09ac\u09be\u0995\u09cd\u09af\u09c7 \u09ac\u09b2\u09cb\u0964\"}\n{\"id\": \"p046\", \"category\": \"indic\", \"lang\": \"bn\", \"prompt\": \"\u09ad\u09be\u09b0\u09a4\u09c7\u09b0 \u09b0\u09be\u099c\u09a7\u09be\u09a8\u09c0 \u0995\u09c0 \u098f\u09ac\u0982 \u09b8\u09c7\u099f\u09bf \u0995\u09cb\u09a8 \u09a8\u09a6\u09c0\u09b0 \u0995\u09be\u099b\u09c7 \u0985\u09ac\u09b8\u09cd\u09a5\u09bf\u09a4?\"}\n{\"id\": \"p047\", \"category\": \"indic\", \"lang\": \"ta\", \"prompt\": \"\u0ba8\u0ba3\u0bcd\u0baa\u0ba9\u0bbf\u0ba9\u0bcd \u0baa\u0bbf\u0bb1\u0ba8\u0bcd\u0ba4\u0ba8\u0bbe\u0bb3\u0bc1\u0b95\u0bcd\u0b95\u0bc1 \u0b92\u0bb0\u0bc1 \u0b9a\u0bbf\u0bb1\u0bbf\u0baf \u0bb5\u0bbe\u0bb4\u0bcd\u0ba4\u0bcd\u0ba4\u0bc1 \u0b8e\u0bb4\u0bc1\u0ba4\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd.\"}\n{\"id\": \"p048\", \"category\": \"indic\", \"lang\": \"ta\", \"prompt\": \"\u0bae\u0bb4\u0bc8\u0b95\u0bcd\u0b95\u0bbe\u0bb2\u0bae\u0bcd \u0ba8\u0bae\u0b95\u0bcd\u0b95\u0bc1 \u0b8f\u0ba9\u0bcd \u0bae\u0bc1\u0b95\u0bcd\u0b95\u0bbf\u0baf\u0bae\u0bcd? \u0b87\u0bb0\u0ba3\u0bcd\u0b9f\u0bc1 \u0bb5\u0bbe\u0b95\u0bcd\u0b95\u0bbf\u0baf\u0b99\u0bcd\u0b95\u0bb3\u0bbf\u0bb2\u0bcd \u0b9a\u0bca\u0bb2\u0bcd\u0bb2\u0bc1\u0b99\u0bcd\u0b95\u0bb3\u0bcd.\"}\n{\"id\": \"p049\", \"category\": \"indic\", \"lang\": \"ta\", \"prompt\": \"\u0b87\u0ba8\u0bcd\u0ba4\u0bbf\u0baf\u0bbe\u0bb5\u0bbf\u0ba9\u0bcd \u0ba4\u0bb2\u0bc8\u0ba8\u0b95\u0bb0\u0bae\u0bcd \u0b8e\u0ba4\u0bc1, \u0b85\u0ba4\u0bc1 \u0b8e\u0ba8\u0bcd\u0ba4 \u0b86\u0bb1\u0bcd\u0bb1\u0bbf\u0ba9\u0bcd \u0b85\u0bb0\u0bc1\u0b95\u0bbf\u0bb2\u0bcd \u0b89\u0bb3\u0bcd\u0bb3\u0ba4\u0bc1?\"}\n{\"id\": \"p050\", \"category\": \"indic\", \"lang\": \"te\", \"prompt\": \"\u0c38\u0c4d\u0c28\u0c47\u0c39\u0c3f\u0c24\u0c41\u0c21\u0c3f \u0c2a\u0c41\u0c1f\u0c4d\u0c1f\u0c3f\u0c28\u0c30\u0c4b\u0c1c\u0c41\u0c15\u0c41 \u0c12\u0c15 \u0c1a\u0c3f\u0c28\u0c4d\u0c28 \u0c36\u0c41\u0c2d\u0c3e\u0c15\u0c3e\u0c02\u0c15\u0c4d\u0c37\u0c32 \u0c38\u0c02\u0c26\u0c47\u0c36\u0c02 \u0c30\u0c3e\u0c2f\u0c02\u0c21\u0c3f.\"}\n{\"id\": \"p051\", \"category\": \"indic\", \"lang\": \"te\", \"prompt\": \"\u0c35\u0c30\u0c4d\u0c37\u0c3e\u0c15\u0c3e\u0c32\u0c02 \u0c2e\u0c28\u0c15\u0c41 \u0c0e\u0c02\u0c26\u0c41\u0c15\u0c41 \u0c2e\u0c41\u0c16\u0c4d\u0c2f\u0c2e\u0c48\u0c28\u0c26\u0c3f? \u0c30\u0c46\u0c02\u0c21\u0c41 \u0c35\u0c3e\u0c15\u0c4d\u0c2f\u0c3e\u0c32\u0c32\u0c4b \u0c1a\u0c46\u0c2a\u0c4d\u0c2a\u0c02\u0c21\u0c3f.\"}\n{\"id\": \"p052\", \"category\": \"indic\", \"lang\": \"te\", \"prompt\": \"\u0c2d\u0c3e\u0c30\u0c24\u0c26\u0c47\u0c36 \u0c30\u0c3e\u0c1c\u0c27\u0c3e\u0c28\u0c3f \u0c0f\u0c26\u0c3f, \u0c05\u0c26\u0c3f \u0c0f \u0c28\u0c26\u0c3f \u0c12\u0c21\u0c4d\u0c21\u0c41\u0c28 \u0c09\u0c02\u0c26\u0c3f?\"}\n{\"id\": \"p053\", \"category\": \"indic\", \"lang\": \"mr\", \"prompt\": \"\u092e\u093f\u0924\u094d\u0930\u093e\u091a\u094d\u092f\u093e \u0935\u093e\u0922\u0926\u093f\u0935\u0938\u093e\u0938\u093e\u0920\u0940 \u090f\u0915 \u091b\u094b\u091f\u093e \u0936\u0941\u092d\u0947\u091a\u094d\u091b\u093e \u0938\u0902\u0926\u0947\u0936 \u0932\u093f\u0939\u093e.\"}\n{\"id\": \"p054\", \"category\": \"indic\", \"lang\": \"mr\", \"prompt\": \"\u092a\u093e\u0935\u0938\u093e\u0933\u093e \u0906\u092a\u0932\u094d\u092f\u093e\u0938\u093e\u0920\u0940 \u0915\u093e \u092e\u0939\u0924\u094d\u0924\u094d\u0935\u093e\u091a\u093e \u0906\u0939\u0947? \u0926\u094b\u0928 \u0935\u093e\u0915\u094d\u092f\u093e\u0902\u0924 \u0938\u093e\u0902\u0917\u093e.\"}\n{\"id\": \"p055\", \"category\": \"indic\", \"lang\": \"mr\", \"prompt\": \"\u092d\u093e\u0930\u0924\u093e\u091a\u0940 \u0930\u093e\u091c\u0927\u093e\u0928\u0940 \u0915\u094b\u0923\u0924\u0940 \u0906\u0923\u093f \u0924\u0940 \u0915\u094b\u0923\u0924\u094d\u092f\u093e \u0928\u0926\u0940\u091c\u0935\u0933 \u0906\u0939\u0947?\"}\n{\"id\": \"p056\", \"category\": \"indic\", \"lang\": \"kn\", \"prompt\": \"\u0cb8\u0ccd\u0ca8\u0cc7\u0cb9\u0cbf\u0ca4\u0ca8 \u0cb9\u0cc1\u0c9f\u0ccd\u0c9f\u0cc1\u0cb9\u0cac\u0ccd\u0cac\u0c95\u0ccd\u0c95\u0cc6 \u0c92\u0c82\u0ca6\u0cc1 \u0c9a\u0cbf\u0c95\u0ccd\u0c95 \u0cb6\u0cc1\u0cad\u0cbe\u0cb6\u0caf \u0cb8\u0c82\u0ca6\u0cc7\u0cb6 \u0cac\u0cb0\u0cc6\u0caf\u0cbf\u0cb0\u0cbf.\"}\n{\"id\": \"p057\", \"category\": \"indic\", \"lang\": \"kn\", \"prompt\": \"\u0cae\u0cb3\u0cc6\u0c97\u0cbe\u0cb2 \u0ca8\u0cae\u0c97\u0cc6 \u0c8f\u0c95\u0cc6 \u0cae\u0cc1\u0c96\u0ccd\u0caf? \u0c8e\u0cb0\u0ca1\u0cc1 \u0cb5\u0cbe\u0c95\u0ccd\u0caf\u0c97\u0cb3\u0cb2\u0ccd\u0cb2\u0cbf \u0cb9\u0cc7\u0cb3\u0cbf.\"}\n{\"id\": \"p058\", \"category\": \"indic\", \"lang\": \"kn\", \"prompt\": \"\u0cad\u0cbe\u0cb0\u0ca4\u0ca6 \u0cb0\u0cbe\u0c9c\u0ca7\u0cbe\u0ca8\u0cbf \u0caf\u0cbe\u0cb5\u0cc1\u0ca6\u0cc1, \u0c85\u0ca6\u0cc1 \u0caf\u0cbe\u0cb5 \u0ca8\u0ca6\u0cbf\u0caf \u0cac\u0cb3\u0cbf \u0c87\u0ca6\u0cc6?\"}\n{\"id\": \"p059\", \"category\": \"indic\", \"lang\": \"ml\", \"prompt\": \"\u0d38\u0d41\u0d39\u0d43\u0d24\u0d4d\u0d24\u0d3f\u0d28\u0d4d\u0d31\u0d46 \u0d1c\u0d28\u0d4d\u0d2e\u0d26\u0d3f\u0d28\u0d24\u0d4d\u0d24\u0d3f\u0d28\u0d4d \u0d12\u0d30\u0d41 \u0d1a\u0d46\u0d31\u0d3f\u0d2f \u0d06\u0d36\u0d02\u0d38 \u0d0e\u0d34\u0d41\u0d24\u0d41\u0d15.\"}\n{\"id\": \"p060\", \"category\": \"indic\", \"lang\": \"ml\", \"prompt\": \"\u0d2e\u0d34\u0d15\u0d4d\u0d15\u0d3e\u0d32\u0d02 \u0d28\u0d2e\u0d41\u0d15\u0d4d\u0d15\u0d4d \u0d0e\u0d28\u0d4d\u0d24\u0d41\u0d15\u0d4a\u0d23\u0d4d\u0d1f\u0d4d \u0d2a\u0d4d\u0d30\u0d27\u0d3e\u0d28\u0d2e\u0d3e\u0d23\u0d4d? \u0d30\u0d23\u0d4d\u0d1f\u0d4d \u0d35\u0d3e\u0d15\u0d4d\u0d2f\u0d19\u0d4d\u0d19\u0d33\u0d3f\u0d7d \u0d2a\u0d31\u0d2f\u0d41\u0d15.\"}\n{\"id\": \"p061\", \"category\": \"indic\", \"lang\": \"ml\", \"prompt\": \"\u0d07\u0d28\u0d4d\u0d24\u0d4d\u0d2f\u0d2f\u0d41\u0d1f\u0d46 \u0d24\u0d32\u0d38\u0d4d\u0d25\u0d3e\u0d28\u0d02 \u0d0f\u0d24\u0d3e\u0d23\u0d4d, \u0d05\u0d24\u0d4d \u0d0f\u0d24\u0d4d \u0d28\u0d26\u0d3f\u0d15\u0d4d\u0d15\u0d30\u0d3f\u0d15\u0d3f\u0d32\u0d3e\u0d23\u0d4d?\"}\n{\"id\": \"p062\", \"category\": \"indic\", \"lang\": \"gu\", \"prompt\": \"\u0aae\u0abf\u0aa4\u0acd\u0ab0\u0aa8\u0abe \u0a9c\u0aa8\u0acd\u0aae\u0aa6\u0abf\u0ab5\u0ab8 \u0aae\u0abe\u0a9f\u0ac7 \u0a8f\u0a95 \u0aa8\u0abe\u0aa8\u0acb \u0ab6\u0ac1\u0aad\u0ac7\u0a9a\u0acd\u0a9b\u0abe \u0ab8\u0a82\u0aa6\u0ac7\u0ab6 \u0ab2\u0a96\u0acb.\"}\n{\"id\": \"p063\", \"category\": \"indic\", \"lang\": \"gu\", \"prompt\": \"\u0a9a\u0acb\u0aae\u0abe\u0ab8\u0ac1\u0a82 \u0a86\u0aaa\u0aa3\u0abe \u0aae\u0abe\u0a9f\u0ac7 \u0a95\u0ac7\u0aae \u0aae\u0ab9\u0aa4\u0acd\u0ab5\u0aa8\u0ac1\u0a82 \u0a9b\u0ac7? \u0aac\u0ac7 \u0ab5\u0abe\u0a95\u0acd\u0aaf\u0acb\u0aae\u0abe\u0a82 \u0a9c\u0aa3\u0abe\u0ab5\u0acb.\"}\n{\"id\": \"p064\", \"category\": \"indic\", \"lang\": \"gu\", \"prompt\": \"\u0aad\u0abe\u0ab0\u0aa4\u0aa8\u0ac0 \u0ab0\u0abe\u0a9c\u0aa7\u0abe\u0aa8\u0ac0 \u0a95\u0a88 \u0a9b\u0ac7 \u0a85\u0aa8\u0ac7 \u0aa4\u0ac7 \u0a95\u0a88 \u0aa8\u0aa6\u0ac0 \u0aaa\u0abe\u0ab8\u0ac7 \u0a86\u0ab5\u0ac7\u0ab2\u0ac0 \u0a9b\u0ac7?\"}\n{\"id\": \"p065\", \"category\": \"indic\", \"lang\": \"pa\", \"prompt\": \"\u0a06\u0a2a\u0a23\u0a47 \u0a26\u0a4b\u0a38\u0a24 \u0a26\u0a47 \u0a1c\u0a28\u0a2e\u0a26\u0a3f\u0a28 \u0a32\u0a08 \u0a07\u0a71\u0a15 \u0a1b\u0a4b\u0a1f\u0a3e \u0a38\u0a3c\u0a41\u0a2d\u0a15\u0a3e\u0a2e\u0a28\u0a3e \u0a38\u0a41\u0a28\u0a47\u0a39\u0a3e \u0a32\u0a3f\u0a16\u0a4b\u0964\"}\n{\"id\": \"p066\", \"category\": \"indic\", \"lang\": \"pa\", \"prompt\": \"\u0a2c\u0a30\u0a38\u0a3e\u0a24 \u0a26\u0a3e \u0a2e\u0a4c\u0a38\u0a2e \u0a38\u0a3e\u0a21\u0a47 \u0a32\u0a08 \u0a15\u0a3f\u0a09\u0a02 \u0a1c\u0a3c\u0a30\u0a42\u0a30\u0a40 \u0a39\u0a48? \u0a26\u0a4b \u0a35\u0a3e\u0a15\u0a3e\u0a02 \u0a35\u0a3f\u0a71\u0a1a \u0a26\u0a71\u0a38\u0a4b\u0964\"}\n{\"id\": \"p067\", \"category\": \"indic\", \"lang\": \"pa\", \"prompt\": \"\u0a2d\u0a3e\u0a30\u0a24 \u0a26\u0a40 \u0a30\u0a3e\u0a1c\u0a27\u0a3e\u0a28\u0a40 \u0a15\u0a40 \u0a39\u0a48 \u0a05\u0a24\u0a47 \u0a09\u0a39 \u0a15\u0a3f\u0a38 \u0a26\u0a30\u0a3f\u0a06 \u0a26\u0a47 \u0a15\u0a4b\u0a32 \u0a39\u0a48?\"}\n{\"id\": \"p068\", \"category\": \"indic\", \"lang\": \"or\", \"prompt\": \"\u0b2c\u0b28\u0b4d\u0b27\u0b41\u0b30 \u0b1c\u0b28\u0b4d\u0b2e\u0b26\u0b3f\u0b28 \u0b2a\u0b3e\u0b07\u0b01 \u0b17\u0b4b\u0b1f\u0b3f\u0b0f \u0b1b\u0b4b\u0b1f \u0b36\u0b41\u0b2d\u0b47\u0b1a\u0b4d\u0b1b\u0b3e \u0b2c\u0b3e\u0b30\u0b4d\u0b24\u0b4d\u0b24\u0b3e \u0b32\u0b47\u0b16\u0964\"}\n{\"id\": \"p069\", \"category\": \"indic\", \"lang\": \"or\", \"prompt\": \"\u0b2c\u0b30\u0b4d\u0b37\u0b3e\u0b0b\u0b24\u0b41 \u0b06\u0b2e \u0b2a\u0b3e\u0b07\u0b01 \u0b15\u0b3e\u0b39\u0b3f\u0b01\u0b15\u0b3f \u0b17\u0b41\u0b30\u0b41\u0b24\u0b4d\u0b71\u0b2a\u0b42\u0b30\u0b4d\u0b23\u0b4d\u0b23? \u0b26\u0b41\u0b07\u0b1f\u0b3f \u0b2c\u0b3e\u0b15\u0b4d\u0b5f\u0b30\u0b47 \u0b15\u0b41\u0b39\u0964\"}\n{\"id\": \"p070\", \"category\": \"indic\", \"lang\": \"or\", \"prompt\": \"\u0b2d\u0b3e\u0b30\u0b24\u0b30 \u0b30\u0b3e\u0b1c\u0b27\u0b3e\u0b28\u0b40 \u0b15'\u0b23 \u0b0f\u0b2c\u0b02 \u0b24\u0b3e\u0b39\u0b3e \u0b15\u0b47\u0b09\u0b01 \u0b28\u0b26\u0b40 \u0b2a\u0b3e\u0b16\u0b30\u0b47 \u0b05\u0b1b\u0b3f?\"}\n{\"id\": \"p071\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Write a Python function that returns the n-th Fibonacci number using iteration.\"}\n{\"id\": \"p072\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Write a Python function that checks whether a string is a palindrome, ignoring case and spaces.\"}\n{\"id\": \"p073\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Write a SQL query that returns the three most recent orders for each customer from a table orders(customer_id, id, created_at).\"}\n{\"id\": \"p074\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Explain what this does: `sorted(d.items(), key=lambda kv: kv[1], reverse=True)[:3]`\"}\n{\"id\": \"p075\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Write a bash one-liner that counts the lines in every .txt file in the current folder.\"}\n{\"id\": \"p076\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Write a JavaScript function that debounces another function by a given number of milliseconds.\"}\n{\"id\": \"p077\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Write a Python class Stack with push, pop and peek, and raise IndexError when it is empty.\"}\n{\"id\": \"p078\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Fix the bug: `def avg(xs): return sum(xs) / len(xs) + 1`\"}\n{\"id\": \"p079\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Write a regular expression that matches a 10-digit mobile number starting with 6, 7, 8 or 9, and explain it.\"}\n{\"id\": \"p080\", \"category\": \"code\", \"lang\": \"en\", \"prompt\": \"Write a Python function that merges two sorted lists into one sorted list without using sort().\"}\n{\"id\": \"p081\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: What is the weather in Pune right now?\\nAssistant: \"}\n{\"id\": \"p082\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: Remind me in 20 minutes to take the idlis off the stove.\\nAssistant: \"}\n{\"id\": \"p083\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: Find my notes about the rent agreement.\\nAssistant: \"}\n{\"id\": \"p084\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: Put 'dentist' on my calendar for 2026-12-03.\\nAssistant: \"}\n{\"id\": \"p085\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: Is it raining in Chennai?\\nAssistant: \"}\n{\"id\": \"p086\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: Set a timer for ten minutes.\\nAssistant: \"}\n{\"id\": \"p087\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: Look up my notes on the Kannada class.\\nAssistant: \"}\n{\"id\": \"p088\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: Add 'team lunch' on 2026-11-27 to the calendar.\\nAssistant: \"}\n{\"id\": \"p089\", \"category\": \"tool_call\", \"lang\": \"en\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: Tell me a joke.\\nAssistant: \"}\n{\"id\": \"p090\", \"category\": \"tool_call\", \"lang\": \"hi\", \"prompt\": \"You can call these tools by replying with one JSON object {\\\"tool\\\": name, \\\"arguments\\\": {...}} and nothing else.\\nTools: get_weather(city: str), set_timer(minutes: int), search_notes(query: str), add_event(title: str, date: str).\\n\\nUser: \u092e\u0941\u0902\u092c\u0908 \u092e\u0947\u0902 \u0906\u091c \u092e\u094c\u0938\u092e \u0915\u0948\u0938\u093e \u0939\u0948?\\nAssistant: \"}\n{\"id\": \"p091\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"A shop sells pens at 14 rupees each. How much do 23 pens cost? Show the steps.\"}\n{\"id\": \"p092\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"If today is Friday, what day of the week is it 45 days from now? Explain.\"}\n{\"id\": \"p093\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"A tank holds 480 litres and leaks 6 litres per hour. How many hours until it is empty?\"}\n{\"id\": \"p094\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"Which is larger, 7/9 or 5/7? Explain how you decided.\"}\n{\"id\": \"p095\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"Three friends split a bill of 1,260 rupees equally, then each adds a 10% tip on their share. What does each pay in total?\"}\n{\"id\": \"p096\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"Ravi is older than Meena. Meena is older than Kiran. Who is the youngest? Explain.\"}\n{\"id\": \"p097\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"What is 17 multiplied by 24? Show the steps.\"}\n{\"id\": \"p098\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"A recipe needs 250 g flour for 4 people. How much flour for 10 people?\"}\n{\"id\": \"p099\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"How many minutes are there in 3 days? Show the steps.\"}\n{\"id\": \"p100\", \"category\": \"reasoning\", \"lang\": \"en\", \"prompt\": \"Put these in order from smallest to largest: 0.5, 1/3, 0.45, 2/5.\"}\n"
}
for _name, _text in SOURCES.items():
    open(f"{CODE}/{_name}", "w").write(_text)
print("wrote", sorted(SOURCES), "to", CODE)

In [ ]:
# llama-cpp-python with CUDA (compiled here, ~10-25 min). The wheel is kept where it survives a new session
# (Colab: Drive), so a later session installs it in seconds.
# GGML_CUDA_NO_VMM: Kaggle's build image has no libcuda for CMake to link (found by therahulbhati, perplexity/README.md).
#
# The evaluation runs in its OWN venv (EVAL_PY) holding only llama-cpp-python, lm_eval and datasets at G9's pins.
# lm_eval imports torch and transformers whenever it finds them, and Kaggle's preinstalled ones fail to import
# (torch: undefined symbol ncclCommShrink; transformers: a dependency version check). This venv has neither.
if LOCAL_TEST:
    LLAMA_SPEC, UV_PYTHON = f"llama-cpp-python=={LLAMA_CPP_PYTHON}", "3.12"   # uv builds it (Metal on a Mac)
else:
    UV_PYTHON = "python3"   # the platform's python, the one the wheel is built for
    sh("pip install -q uv==0.13.6", log=f"{OUT}/logs/pip.log")
    WHEELS = f"{OUT}/wheels" if OUT == DRIVE_DIR else f"{WORK}/wheels"
    wheels = glob.glob(f"{WHEELS}/llama_cpp_python-{LLAMA_CPP_PYTHON}-*.whl")
    if not wheels:
        os.makedirs(WHEELS, exist_ok=True)
        caps = sorted({c.replace(".", "") for c in subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader",
                       shell=True, capture_output=True, text=True).stdout.split()}) or ["75"]
        ENV["cuda_arch"] = caps; save_env()
        # -v makes the build print ninja's [done/total] counter; the full output goes to the log, the counter here
        build_log = f"{OUT}/logs/build_llama_cpp_python.log"
        print("compiling llama-cpp-python with CUDA: ~15-30 min; progress every ~20 kernels below")
        sh(f'CMAKE_ARGS="-DGGML_CUDA=on -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES={";".join(caps)}" '
           f'pip wheel -v --no-deps --no-cache-dir -w {WHEELS} llama-cpp-python=={LLAMA_CPP_PYTHON} 2>&1 '
           f'| tee {build_log} | grep --line-buffered -E "^ *[[][0-9]+/[0-9]+[]]" | awk "NR % 20 == 1 {{print; fflush()}}" ; '
           f'ls {WHEELS}/llama_cpp_python-*.whl >/dev/null || (tail -40 {build_log}; false)')
        wheels = glob.glob(f"{WHEELS}/llama_cpp_python-{LLAMA_CPP_PYTHON}-*.whl")
    LLAMA_SPEC = wheels[0]

EVAL_VENV = f"{WORK}/eval-venv"
EVAL_PY = f"env -u PYTHONPATH {EVAL_VENV}/bin/python"   # also without Kaggle's PYTHONPATH sitecustomize hook
if not os.path.exists(f"{EVAL_VENV}/.ready"):
    sh(f"rm -rf {EVAL_VENV} && uv venv -q --python {UV_PYTHON} {EVAL_VENV} && uv pip install -q --python {EVAL_VENV}/bin/python "
       f"{LLAMA_SPEC} lm_eval==0.4.13 datasets==5.1.0 && touch {EVAL_VENV}/.ready", log=f"{OUT}/logs/eval_venv.log")
probe = f"{CODE}/probe_env.py"
open(probe, "w").write(os.linesep.join([
    "import importlib.util, json, llama_cpp, lm_eval, datasets, platform",
    "print(json.dumps({'python': platform.python_version(), 'llama_cpp_python': llama_cpp.__version__,",
    "  'lm_eval': lm_eval.__version__, 'datasets': datasets.__version__,",
    "  'gpu_offload': bool(llama_cpp.llama_supports_gpu_offload()),",
    "  'torch_visible': importlib.util.find_spec('torch') is not None,",
    "  'transformers_visible': importlib.util.find_spec('transformers') is not None}))", ""]))
res = subprocess.run(f"{EVAL_PY} {probe}", shell=True, capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError("the evaluation venv does not work: " + res.stderr[-3000:])
ENV["packages"] = json.loads(res.stdout.strip().splitlines()[-1])
if not LOCAL_TEST and not ENV["packages"]["gpu_offload"]:
    raise RuntimeError("llama-cpp-python has no GPU offload: delete the wheels folder and rerun to rebuild it with CUDA")
if ENV["packages"]["torch_visible"] or ENV["packages"]["transformers_visible"]:
    raise RuntimeError(f"torch/transformers visible in the evaluation venv: {ENV['packages']}")
save_env(); print(ENV["packages"])

In [ ]:
# bf16 GGUF: the llama.cpp converter at the team's build, pinned checkpoint; llama-quantize built CPU-only (~5 min)
NAME = MODEL.split("/")[-1]
BF16 = f"{WORK}/{NAME}-bf16.gguf"
QUANTIZE = shutil.which("llama-quantize") if LOCAL_TEST else f"{WORK}/llama.cpp/build/bin/llama-quantize"
if not LOCAL_TEST and not os.path.exists(QUANTIZE):
    if not os.path.exists(f"{WORK}/llama.cpp"):
        sh(f"git clone -q --depth 1 --branch {LLAMA_TAG} https://github.com/ggml-org/llama.cpp {WORK}/llama.cpp")
    sh(f"cmake -S {WORK}/llama.cpp -B {WORK}/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF "
       f"&& cmake --build {WORK}/llama.cpp/build -j --target llama-quantize", log=f"{OUT}/logs/build_llama_quantize.log")
if not os.path.exists(BF16):
    # the converter's own requirements (pinned torch, numpy, ...) go into a separate venv, so they cannot replace
    # packages this notebook has already imported
    CONV = f"{WORK}/convert-venv"
    if not os.path.exists(f"{CONV}/.ready"):   # uv: Kaggle's python has no ensurepip, so plain venv fails
        sh(f"rm -rf {CONV} && pip install -q uv==0.13.6 && uv venv -q --python python3 {CONV} "
           f"&& uv pip install -q --python {CONV}/bin/python --index-strategy unsafe-best-match "
           f"-r {WORK}/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt {WORK}/llama.cpp/gguf-py "
           f"&& touch {CONV}/.ready", log=f"{OUT}/logs/convert_venv.log")
    from huggingface_hub import snapshot_download
    hf = snapshot_download(MODEL, revision=REVISION, local_dir=f"{WORK}/hf",
                           allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"])
    # env -u PYTHONPATH: Kaggle injects a sitecustomize hook through PYTHONPATH that needs packages this venv lacks
    sh(f"env -u PYTHONPATH {CONV}/bin/python {WORK}/llama.cpp/convert_hf_to_gguf.py {hf} --outtype bf16 --outfile {BF16}",
       log=f"{OUT}/logs/convert.log")
    shutil.rmtree(f"{WORK}/hf", ignore_errors=True)
ENV["bf16_gguf_bytes"] = os.path.getsize(BF16); save_env()
print(BF16, round(os.path.getsize(BF16) / 1e9, 2), "GB")

In [ ]:
# every format: quantize (output layer Q8_0 below 8 bits, as quantize_eval.py), rollouts + scoring, one JSON per format
RUNS = f"{OUT}/rollout_{NAME}_n{N}_k{K}_t{MAX_NEW}" + ("_balanced" if PROMPTS == "balanced" else "")
os.makedirs(RUNS, exist_ok=True)

def run_format(fmt, gguf, ref_layers):
    out = f"{RUNS}/{fmt}.json"
    sh(f"cd {CODE} && {EVAL_PY} quant_rollout_mismatch.py --ref {BF16} --quant {fmt}={gguf} --prompts {CODE}/fixed_100.jsonl "
       f"--out {out} --n {N} --k {K} --max-new {MAX_NEW} --n-ctx {N_CTX} --eps {EPS} --seed {SEED} "
       f"--ref-n-gpu-layers {ref_layers} --dump-dir {RUNS}/dump" + (" --balanced" if PROMPTS == "balanced" else ""), log=f"{OUT}/logs/rollout_{fmt}.log")

for fmt in FORMATS:
    t0 = time.time()
    if os.path.exists(f"{RUNS}/{fmt}.json"):
        print(fmt, "already done, skipping"); continue
    gguf = BF16 if fmt == "bf16" else f"{WORK}/{NAME}-{fmt}.gguf"
    made_here = fmt != "bf16" and not os.path.exists(gguf)       # the bf16 file is the reference: never made or deleted here
    if made_here:
        extra = "" if fmt == "Q8_0" else "--output-tensor-type q8_0"
        sh(f"{QUANTIZE} {extra} {BF16} {gguf} {fmt}", log=f"{OUT}/logs/quantize_{fmt}.log")
    ref_layers = -1
    try:
        run_format(fmt, gguf, ref_layers)
    except RuntimeError:
        tail = open(f"{OUT}/logs/rollout_{fmt}.log").read()[-20000:].lower()
        if not any(k in tail for k in ("out of memory", "failed to allocate", "cudamalloc")):
            raise
        ref_layers = BF16_FALLBACK_GPU_LAYERS
        print(f"out of GPU memory; retrying with the bf16 reference on {ref_layers} GPU layers")
        run_format(fmt, gguf, ref_layers)
    ENV["formats"][fmt] = {"ref_gpu_layers": ref_layers, "gguf_bytes": os.path.getsize(gguf),
                           "wall_clock_s": round(time.time() - t0), "finished": datetime.datetime.now().isoformat(timespec="seconds")}
    save_env()
    if made_here:
        os.remove(gguf)

In [ ]:
# stage 2: the same sampled tokens scored by the Hugging Face model in PyTorch (Team 10's setup); pinned venv, GPU must be visible
TORCH_JSON = f"{RUNS}/torch_reference.json"
if TORCH_REFERENCE and not os.path.exists(TORCH_JSON):
    TVENV = f"{WORK}/torch-venv"; TPY = f"env -u PYTHONPATH {TVENV}/bin/python"
    if not os.path.exists(f"{TVENV}/.ready"):
        sh(f"rm -rf {TVENV} && pip install -q uv==0.13.6 && uv venv -q --python python3 {TVENV} && "
           f"uv pip install -q --python {TVENV}/bin/python {TORCH} --index-url {TORCH_INDEX} && "
           f"uv pip install -q --python {TVENV}/bin/python transformers==4.57.6 numpy accelerate && touch {TVENV}/.ready",
           log=f"{OUT}/logs/torch_venv.log")
    pr = subprocess.run(f"{TPY} -c \"import torch, json; print(json.dumps(dict(torch=torch.__version__, cuda=torch.cuda.is_available())))\"",
                        shell=True, capture_output=True, text=True)
    if pr.returncode != 0 or not json.loads(pr.stdout.strip().splitlines()[-1])["cuda"]:
        raise RuntimeError("the torch venv cannot see the GPU (" + (pr.stderr[-500:] or pr.stdout) + "): set TORCH_INDEX to "
                           "https://download.pytorch.org/whl/cu128, delete " + TVENV + " and rerun")
    ENV["torch_venv"] = json.loads(pr.stdout.strip().splitlines()[-1]); save_env()
    dumps = " ".join(f"--dump {RUNS}/dump/{f}_rollouts.jsonl" for f in FORMATS if os.path.exists(f"{RUNS}/dump/{f}_rollouts.jsonl"))
    rev = f"--revision {REVISION} " if REVISION else ""
    sh(f"cd {CODE} && env HF_HOME={WORK}/hf {TPY} -u score_with_torch.py {dumps} --model {MODEL} {rev}--out {TORCH_JSON} "
       f"--dtype {TORCH_DTYPE} --eps {EPS} --device cuda", log=f"{OUT}/logs/score_with_torch.log")

In [ ]:
# results: one table over the formats, a zip of everything
import pandas as pd
rows = []
for p in sorted(glob.glob(f"{RUNS}/*.json")):
    if os.path.basename(p) in ("torch_reference.json", "environment.json"):   # not per-format llama.cpp results
        continue
    d = json.load(open(p))
    for f in d["formats"]:
        rows.append({"format": f["format"], "rollouts": f["rollouts"], "new_tokens": f["new_tokens"],
                     "rollout_tok_s": f["rollout_tokens_per_s"], "outside": round(f.get("outside", float("nan")), 4),
                     "below": round(f.get("below", float("nan")), 4), "above": round(f.get("above", float("nan")), 4),
                     "mean_abs_logratio": round(f.get("mean_abs_logratio", float("nan")), 4),
                     "k3": round(f.get("k3", float("nan")), 5), "kl_bf16_to_quant": round(f.get("mean_kl_ref_to_quant", float("nan")), 5),
                     "top1_agree": round(f.get("top1_agreement", float("nan")), 4)})
table = pd.DataFrame(rows)
if (table["format"] == "bf16").any():       # the real speed ratio: this sampler against the bf16 sampler, same prompts, same session
    base_speed = float(table.loc[table["format"] == "bf16", "rollout_tok_s"].iloc[0])
    table["tok_s_vs_bf16"] = (table["rollout_tok_s"] / base_speed).round(2)
table.to_csv(f"{RUNS}/table.csv", index=False)
print("reference = llama.cpp bf16 GGUF"); print(table.to_string(index=False))
if os.path.exists(f"{RUNS}/torch_reference.json"):
    tj = json.load(open(f"{RUNS}/torch_reference.json"))
    trows = [{"format": f["format"], "outside": round(f["outside"], 4), "outside_ci95": f["ci95_by_prompt"]["outside"],
              "mean_abs_logratio": round(f["mean_abs_logratio"], 4), "k3": round(f["k3"], 5), "k3_ci95": f["ci95_by_prompt"]["k3"],
              "top1_agree": round(f["top1_agreement"], 4), "top1_ci95": f["ci95_by_prompt"]["top1_agreement"]} for f in tj["formats"]]
    ttable = pd.DataFrame(trows); ttable.to_csv(f"{RUNS}/table_torch.csv", index=False)
    print(f"\nreference = PyTorch {tj['dtype']} ({tj['reference']}); intervals resample prompts"); print(ttable.to_string(index=False))
ENV["finished"] = datetime.datetime.now().isoformat(timespec="seconds"); save_env()
shutil.copy(f"{OUT}/environment.json", f"{RUNS}/environment.json")
shutil.copytree(f"{OUT}/logs", f"{RUNS}/logs", dirs_exist_ok=True)
zip_path = shutil.make_archive(RUNS, "zip", RUNS)   # the dump (token ids) is inside: do not commit it
print("\nEverything is in", RUNS, "and", zip_path, "(Kaggle: the version's Output tab)")